# Домашнее задание 7. Сборка конвейера CI/CD

Автор: Вольхин Сергей. Репозиторий: https://github.com/SergeiVolkhin/ML-HW_7

Локальная разработка велась на Windows 11 + Docker Desktop 29.1.3. Полный набор команд для воспроизведения - в README.md проекта.

## 1. Настроить CI/CD-пайплайн для ML-сервиса с использованием GitLab

В пайплайне сохраняются артефакты для воспроизводимости:

- pip freeze (точные версии всех зависимостей)
- git SHA коммита (хеш кода)
- sha256 датасета (через `load_iris().data.tobytes()`)
- sha256 файла обученной модели
- random_state и гиперпараметры RandomForest

Ниже выведен переработанный `.gitlab-ci.yml` (исходный шаблон был в синтаксисе Gitea Actions, переписан под нативный GitLab CI: `stages`, `image`, `script`).

In [ ]:
%%sh
git config --global user.email "twinslolipop@gmail.com"
git config --global user.name "Sergei Volkhin"
git init
pip install scikit-learn numpy pandas fastapi uvicorn joblib pydantic -qqq
pip freeze > requirements.txt

In [ ]:
%%writefile ml_pipeline.py
import argparse
import json
import logging
from pathlib import Path

import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("train")

DEFAULT_HYPERPARAMS = {"n_estimators": 100, "max_depth": None}


def train(output_path: Path, random_state: int, hyperparams: dict) -> dict:
    iris = load_iris()
    x_train, x_test, y_train, y_test = train_test_split(
        iris.data, iris.target, test_size=0.2, random_state=random_state, stratify=iris.target
    )

    model = RandomForestClassifier(random_state=random_state, **hyperparams)
    model.fit(x_train, y_train)

    train_accuracy = accuracy_score(y_train, model.predict(x_train))
    test_accuracy = accuracy_score(y_test, model.predict(x_test))

    output_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, output_path)

    logger.info("train accuracy: %.4f", train_accuracy)
    logger.info("test accuracy: %.4f", test_accuracy)
    logger.info("model saved to %s", output_path)

    return {
        "train_accuracy": train_accuracy,
        "test_accuracy": test_accuracy,
        "random_state": random_state,
        "hyperparameters": hyperparams,
        "n_train": len(x_train),
        "n_test": len(x_test),
    }


def main() -> None:
    parser = argparse.ArgumentParser(description="Train RandomForest on iris dataset")
    parser.add_argument(
        "--output-path",
        type=Path,
        default=Path("app/models/model.pkl"),
        help="Куда сохранить обученную модель",
    )
    parser.add_argument(
        "--random-state",
        type=int,
        default=42,
        help="Random seed для воспроизводимости",
    )
    parser.add_argument(
        "--metrics-path",
        type=Path,
        default=None,
        help="Опциональный JSON с метриками обучения",
    )
    args = parser.parse_args()

    metrics = train(args.output_path, args.random_state, DEFAULT_HYPERPARAMS)

    if args.metrics_path is not None:
        args.metrics_path.parent.mkdir(parents=True, exist_ok=True)
        args.metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()


### Проверка работоспособности пайплайна обучения

In [ ]:
!python ml_pipeline.py --output-path artifacts/model.pkl --random-state 42

In [ ]:
%%writefile .gitlab-ci.yml
stages:
  - lint
  - test
  - train
  - build
  - reproducibility

default:
  image: python:3.11-slim
  before_script:
    - python -m pip install --upgrade pip
    - pip install -r requirements-dev.txt
  cache:
    key:
      files:
        - requirements.txt
        - requirements-dev.txt
    paths:
      - .cache/pip

variables:
  PIP_CACHE_DIR: "$CI_PROJECT_DIR/.cache/pip"
  PYTHONDONTWRITEBYTECODE: "1"

lint:
  stage: lint
  script:
    - ruff check app/ tests/

pytest:
  stage: test
  script:
    - pytest tests/ -v --tb=short

train:
  stage: train
  script:
    - mkdir -p artifacts
    - python -m app.ml_pipeline --output-path artifacts/model.pkl --random-state 42 --metrics-path artifacts/metrics.json
  artifacts:
    paths:
      - artifacts/model.pkl
      - artifacts/metrics.json
    expire_in: 1 week

build:
  stage: build
  image: docker:27
  services:
    - docker:27-dind
  before_script: []
  variables:
    DOCKER_TLS_CERTDIR: ""
    DOCKER_HOST: tcp://docker:2375
  rules:
    - if: $CI_COMMIT_BRANCH == "main"
    - if: $CI_COMMIT_TAG
  script:
    - docker build -f docker/Dockerfile --build-arg MODEL_VERSION=${CI_COMMIT_TAG:-v1.0.0} -t ml-service:${CI_COMMIT_SHORT_SHA} .

# Make pipeline reproducible.
# Сохраняем артефакт pipeline_metadata.json со всем что нужно для повторного запуска
# обучения с теми же результатами:
#   - точные версии всех Python-пакетов (pip freeze)
#   - git SHA коммита, на котором запущен пайплайн (хеш кода)
#   - sha256 от сырых байтов датасета (хеш данных)
#   - sha256 от обученной модели (хеш артефакта)
#   - random_state и гиперпараметры RandomForest
# Этот файл позволяет на любом окружении воспроизвести модель с теми же
# accuracy и predict-выходами.
reproducibility:
  stage: reproducibility
  needs:
    - job: train
      artifacts: true
  script:
    - mkdir -p artifacts
    - pip freeze > artifacts/requirements_lock.txt
    - |
      python - <<'PY'
      import hashlib
      import json
      import os
      import subprocess
      from sklearn.datasets import load_iris

      dataset_hash = hashlib.sha256(load_iris().data.tobytes()).hexdigest()

      with open("artifacts/model.pkl", "rb") as fh:
          model_hash = hashlib.sha256(fh.read()).hexdigest()

      with open("artifacts/metrics.json", "r", encoding="utf-8") as fh:
          metrics = json.load(fh)

      metadata = {
          "code_commit": os.environ.get("CI_COMMIT_SHA", "local"),
          "pipeline_id": os.environ.get("CI_PIPELINE_ID", "local"),
          "dataset_sha256": dataset_hash,
          "model_sha256": model_hash,
          "random_state": metrics["random_state"],
          "hyperparameters": metrics["hyperparameters"],
          "train_accuracy": metrics["train_accuracy"],
          "test_accuracy": metrics["test_accuracy"],
          "requirements_lock": "artifacts/requirements_lock.txt",
      }

      with open("artifacts/pipeline_metadata.json", "w", encoding="utf-8") as fh:
          json.dump(metadata, fh, indent=2, ensure_ascii=False)

      print(json.dumps(metadata, indent=2, ensure_ascii=False))
      PY
  artifacts:
    paths:
      - artifacts/pipeline_metadata.json
      - artifacts/requirements_lock.txt
    expire_in: 1 month


In [ ]:
!git add .gitlab-ci.yml ml_pipeline.py
!git commit -m "ci(gitlab): add GitLab CI pipeline with reproducibility job"
!git log --oneline

### Список коммитов в проекте

```
git log --oneline:

6086183 docs: add README with run and deployment instructions
85e3a33 docs(adr): record A/B testing plan for v1.1.0
50dfe60 docs(adr): record canary deployment strategy with comparison
cfdf2ad ci(github): add GitHub Actions ci and deploy workflows
0122fa2 ci(gitlab): add GitLab CI pipeline with reproducibility job
950f556 feat(canary): add nginx split-clients canary deployment
bf9ad54 build(docker): add Dockerfile and blue/green compose stack
49473e2 test: add pytest coverage for health and predict endpoints
7ddae78 fix(schemas): disable pydantic protected_namespaces for HealthResponse
c304cae feat(ml): add training script for RandomForest on iris
678ab7d feat(app): add FastAPI service with /health and /predict
6cc8b1f chore: initial project structure
```

### Ссылка на успешный пайплайн GitLab

https://gitlab.com/twinslolipop/ml-hw_7/-/pipelines/2516826220


## 2. Обосновать стратегию деплоя (Blue-Green, Canary, Rolling, Shadow) и оценить влияние на риски

ADR сделан по формату Michael Nygard (`adr-tools`). Файлы в `doc/architecture/decisions/`. Ниже - полный текст ADR-0002.

# 2. Использовать Canary deployment для ml-сервиса

Date: 2026-05-11

## Status

Accepted

## Context

В коде ml-сервиса отсутствует системная обработка ошибок (в исходной версии
`app/main.py` логика инференса не была обёрнута в try/except, а валидация входа
делалась только pydantic-схемой). Выкатываем новую версию модели v1.1.0,
обученную на iris RandomForest. На train accuracy достигла 1.0000, что
подозрительно: либо переобучение, либо утечка через стратифицированный split
(см. метрики в `pipeline_metadata.json`).

Требования:
- ограничить долю пользователей, попадающих на потенциально проблемную версию;
- иметь возможность отката за секунды, а не минуты;
- собрать live-метрики качества инференса до полного раскатывания;
- не тратить лишних 100% инфраструктуры под двойной набор сервисов 24/7.

## Decision

Используем Canary deployment с nginx `split_clients` и поэтапным увеличением
веса 10 -> 25 -> 50 -> 100 (см. `scripts/deploy_canary.sh`). Между этапами
60 секунд паузы и health-check обоих апстримов. При неудаче автоматически
вызывается `scripts/rollback_canary.sh`.

### Сравнение альтернатив

| Критерий                            | Blue-Green   | Canary           | Rolling           | Shadow         |
|-------------------------------------|--------------|------------------|-------------------|----------------|
| Downtime при выкатке                | 0            | 0                | 0                 | 0              |
| Скорость отката                     | секунды      | секунды          | минуты            | n/a (без прода)|
| Доля затронутых при баге            | до 100%      | 10% (canary)     | 10-30% (rolling)  | 0% (теневая)   |
| Расход инфры на время деплоя        | 2x           | 1.1x (10% canary)| 1.1-1.3x          | 2x             |
| Сбор live-метрик новой модели       | после switch | сразу на canary  | смешано           | да, но без A/B |
| Сложность реализации                | низкая       | средняя          | высокая (lb-aware)| средняя        |
| Подходит для ML без error-handling  | нет (риск 100%)| да (риск 10%)  | частично          | да             |

Blue-Green переключает 100% трафика мгновенно, что при необработанном
исключении в новой модели приведёт к 100% ошибок и потере доверия. Rolling
подходит для stateless API, но требует поддержки на уровне load balancer
(weighted upstream с health-aware draining) - в нашем nginx без plus это
сложно. Shadow позволяет проверить inference, но не покажет UX-метрики
(latency, ошибки клиента) и не даёт прогрессивного нарастания нагрузки.

Canary даёт лучший компромисс: 10% трафика - это ограниченный blast radius,
при этом метрики собираются на реальной нагрузке.

### Rollback процедура и измеренное время

Откат: `bash scripts/rollback_canary.sh`. Скрипт подменяет nginx.conf на
`nginx.rollback.conf` (без `split_clients`, 100% на app-stable) и вызывает
`nginx -s reload`. Локальный замер показал **439 мс** от начала команды до
завершения reload (нагрузка не сбрасывается, keep-alive соединения остаются
живыми).

После rollback `smoke_test.sh` 100/100 запросов попадает на v1.0.0 - проверено
локально (см. вывод в README).

## Consequences

Положительные:
- blast radius ограничен 10% до получения метрик;
- live-сравнение качества v1.0.0 и v1.1.0 на одном и том же входном трафике;
- откат за секунды без перезапуска контейнеров.

Отрицательные:
- одновременно работают две версии модели - нужно либо stateless API
  (у нас так), либо sticky-сессии. nginx `split_clients` с `$request_id`
  в ключе ломает sticky, поэтому при наличии состояния потребуется правка
  ключа на `$remote_addr` или introducing user-id cookie;
- мониторинг должен различать метрики по версии (используем заголовок
  `X-Service-Version`, проставляемый nginx-ом);
- nginx `split_clients` не принимает 0% как значение веса, поэтому для отката
  используется отдельный статический конфиг.


## 3. Реализовать стратегию развертывания

Реализация Canary через `nginx split_clients` и docker-compose стек из трёх сервисов: app-stable (v1.0.0), app-canary (v1.1.0), nginx.

Ниже - `docker-compose.canary.yml` и шаблон `nginx.canary.conf.template`.

In [ ]:
%%writefile docker-compose.canary.yml
services:
  app-stable:
    image: ml-service:v1.0.0
    container_name: ml-service-stable
    build:
      context: ..
      dockerfile: docker/Dockerfile
      args:
        MODEL_VERSION: v1.0.0
    environment:
      MODEL_VERSION: v1.0.0
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request,sys; sys.exit(0 if urllib.request.urlopen('http://127.0.0.1:8000/health',timeout=2).status==200 else 1)"]
      interval: 10s
      timeout: 3s
      retries: 3
      start_period: 15s
    networks:
      - ml-net

  app-canary:
    image: ml-service:v1.1.0
    container_name: ml-service-canary
    build:
      context: ..
      dockerfile: docker/Dockerfile
      args:
        MODEL_VERSION: v1.1.0
    environment:
      MODEL_VERSION: v1.1.0
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request,sys; sys.exit(0 if urllib.request.urlopen('http://127.0.0.1:8000/health',timeout=2).status==200 else 1)"]
      interval: 10s
      timeout: 3s
      retries: 3
      start_period: 15s
    networks:
      - ml-net

  nginx:
    image: nginx:1.27-alpine
    container_name: ml-nginx
    depends_on:
      app-stable:
        condition: service_healthy
      app-canary:
        condition: service_healthy
    ports:
      - "8080:80"
    environment:
      CANARY_WEIGHT: "${CANARY_WEIGHT:-10}"
    volumes:
      - ./nginx:/templates:ro
    entrypoint: ["/bin/sh", "-c"]
    # envsubst подставляет CANARY_WEIGHT, переменная DOLLAR оставляет $ для nginx
    command:
      - |
        apk add --no-cache gettext >/dev/null
        export DOLLAR='$$'
        envsubst '$$DOLLAR $$CANARY_WEIGHT' \
          < /templates/nginx.canary.conf.template \
          > /etc/nginx/nginx.conf
        echo "[nginx] config generated, CANARY_WEIGHT=$$CANARY_WEIGHT"
        exec nginx -g 'daemon off;'
    networks:
      - ml-net

networks:
  ml-net:
    driver: bridge


In [ ]:
%%writefile nginx.canary.conf.template
worker_processes auto;
events {
    worker_connections 1024;
}

http {
    log_format canary '$remote_addr - $upstream_addr [$time_local] '
                     '"$request" $status $body_bytes_sent '
                     '"$http_user_agent" version=$service_version';
    access_log /var/log/nginx/access.log canary;

    upstream stable {
        server app-stable:8000;
    }

    upstream canary {
        server app-canary:8000;
    }

    # split_clients использует MurmurHash2 от ключа.
    # request_id уникален per-request, поэтому даже curl с одного IP даст
    # ожидаемое распределение веса.
    split_clients "${DOLLAR}remote_addr${DOLLAR}http_user_agent${DOLLAR}request_id" ${DOLLAR}target_pool {
        ${CANARY_WEIGHT}%   canary;
        *                   stable;
    }

    map ${DOLLAR}target_pool ${DOLLAR}service_version {
        canary  v1.1.0;
        stable  v1.0.0;
    }

    server {
        listen 80;

        location /health {
            proxy_pass http://${DOLLAR}target_pool/health;
            proxy_set_header Host ${DOLLAR}host;
            proxy_set_header X-Real-IP ${DOLLAR}remote_addr;
            proxy_set_header X-Forwarded-Version ${DOLLAR}service_version;
            add_header X-Service-Version ${DOLLAR}service_version always;
        }

        location / {
            proxy_pass http://${DOLLAR}target_pool;
            proxy_set_header Host ${DOLLAR}host;
            proxy_set_header X-Real-IP ${DOLLAR}remote_addr;
            proxy_set_header X-Forwarded-For ${DOLLAR}proxy_add_x_forwarded_for;
            proxy_set_header X-Forwarded-Version ${DOLLAR}service_version;
            add_header X-Service-Version ${DOLLAR}service_version always;
        }
    }
}


## 4. Спланировать A/B-тестирование для ML-модели

# 3. План A/B-тестирования модели v1.1.0

Date: 2026-05-11

## Status

Proposed

## Context

После прохождения canary-этапа (10 -> 25 -> 50 -> 100, см. ADR-0002) хочется
получить статистически значимый ответ: новая модель v1.1.0 не хуже текущей
v1.0.0 на проде. Для этого нужен полноценный A/B-эксперимент на размеченных
данных, а не только канареечный health-check.

## Decision

### Гипотезы

- H0: accuracy v1.1.0 = accuracy v1.0.0
- H1: accuracy v1.1.0 != accuracy v1.0.0 (двусторонний тест)

### Метрики

- Primary: accuracy на размеченных данных (ground truth собирается через
  выборочную ручную разметку прогнозов выгружаемых раз в сутки).
- Secondary: latency p95 ответа `POST /predict`, доля 5xx ответов.

### Группы

- A (control, v1.0.0): 50%
- B (treatment, v1.1.0): 50%

После успешного canary с 100% на v1.1.0 переходим в A/B-режим: переключаем
nginx на конфиг с равными весами в `split_clients` (50/50).

### Единица рандомизации

`user_id`, хэшируемый через sha256 и приводимый по модулю к `[0, 100)`. Это
обеспечивает sticky-распределение: один пользователь всегда попадает в одну
группу. Если `user_id` нет, фолбэк - `request_id` (без sticky, но достаточно
для пилотного эксперимента).

### Расчёт sample size

Используем нормальное приближение для пропорций. Параметры:

- alpha = 0.05 (z_alpha/2 = 1.96, двусторонний)
- beta = 0.20 (z_beta = 0.84, power = 80%)
- baseline accuracy p1 = 0.95
- MDE delta = 0.02 -> p2 = 0.97

Формула:

```
n = (z_a/2 + z_b)^2 * (p1*(1-p1) + p2*(1-p2)) / (p2 - p1)^2
  = (1.96 + 0.84)^2 * (0.0475 + 0.0291) / 0.0004
  = 7.84 * 0.0766 / 0.0004
  = 1500.86
```

Округляем до **n = 1501 на группу, всего 3002 размеченных наблюдения**.

Python-проверка:

```python
import math
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

effect = proportion_effectsize(0.97, 0.95)
n = NormalIndPower().solve_power(effect_size=effect, alpha=0.05, power=0.8, alternative="two-sided")
print(math.ceil(n))  # 1471
```

`statsmodels` даёт 1471 на группу (pooled-variance), наша формула - 1501
(unpooled). Закладываем 1501 как более консервативное число.

### Длительность

Минимум 1 неделя, чтобы учесть недельную сезонность (бизнес-дни vs выходные).
При прод-трафике 500 размеченных предсказаний в день получаем 3500 за неделю -
этого достаточно для 3002 наблюдений.

### Критерии остановки

- Положительная остановка: достигли n=1501 в каждой группе, проводим
  `proportions_ztest`. Если p-value < 0.05 и доверительный интервал разницы
  не содержит 0 - принимаем H1, раскатываем v1.1.0 на 100%.
- Раннее прерывание (safety): останавливаем эксперимент если на любом окне
  в 200 запросов latency p95 v1.1.0 деградировала более чем на 50%, или доля
  5xx ошибок превысила 1%.
- Inconclusive: достигли sample size но p-value >= 0.05 - оставляем v1.0.0,
  заводим тикет на улучшение модели.

### Финальный статистический тест

```python
from statsmodels.stats.proportion import proportions_ztest

successes = [n_correct_v100, n_correct_v110]
n_obs = [n_total_v100, n_total_v110]
z_stat, p_value = proportions_ztest(successes, n_obs, alternative="two-sided")
print(f"z={z_stat:.4f}, p={p_value:.4f}")
```

## Consequences

Положительные:
- объективный ответ "выкатываем или откатываем" с заданной statistical power;
- safety-guard рано прерывает эксперимент при латентности/ошибках;
- sticky-распределение через хэш user_id даёт корректную единицу анализа.

Отрицательные:
- нужна разметка прогнозов в проде (выборка с ручной валидацией) - это
  дополнительная нагрузка на команду разметки;
- 50/50 split на неделю означает, что половина пользователей видит новую
  модель до её формальной валидации;
- если фактический baseline accuracy окажется ниже 0.95, sample size
  пересчитывается в сторону увеличения.


In [ ]:
import math

z_alpha_2 = 1.96   # alpha = 0.05, two-sided
z_beta = 0.84      # power = 0.80
p1 = 0.95           # baseline accuracy v1.0.0
p2 = 0.97           # ожидаемая accuracy v1.1.0
mde = p2 - p1       # 0.02

numerator = (z_alpha_2 + z_beta) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2))
denominator = mde ** 2
n_per_group = math.ceil(numerator / denominator)
print(f'sample size per group: {n_per_group}')
print(f'total observations:    {n_per_group * 2}')

In [ ]:
# Проверка статистической значимости результата эксперимента (заглушка).
# После сбора n=1501 в каждой группе и подсчёта числа верных
# предсказаний, вызывается proportions_ztest:

from statsmodels.stats.proportion import proportions_ztest

successes_v100 = 1426   # пример: 1426 из 1501 верны
successes_v110 = 1456   # пример: 1456 из 1501 верны

z_stat, p_value = proportions_ztest(
    [successes_v100, successes_v110],
    [1501, 1501],
    alternative='two-sided',
)
print(f'z = {z_stat:.4f}, p = {p_value:.4f}')
print('reject H0' if p_value < 0.05 else 'fail to reject H0')

## 5. Создать CI/CD-пайплайн для ML-сервиса с использованием GitHub Actions

Часть `Make pipeline reproducible` дописана отдельным шагом в job `train` (см. `.github/workflows/ci.yml`).

In [ ]:
%%writefile ml_pipeline.py
import argparse
import json
import logging
from pathlib import Path

import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("train")

DEFAULT_HYPERPARAMS = {"n_estimators": 100, "max_depth": None}


def train(output_path: Path, random_state: int, hyperparams: dict) -> dict:
    iris = load_iris()
    x_train, x_test, y_train, y_test = train_test_split(
        iris.data, iris.target, test_size=0.2, random_state=random_state, stratify=iris.target
    )

    model = RandomForestClassifier(random_state=random_state, **hyperparams)
    model.fit(x_train, y_train)

    train_accuracy = accuracy_score(y_train, model.predict(x_train))
    test_accuracy = accuracy_score(y_test, model.predict(x_test))

    output_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, output_path)

    logger.info("train accuracy: %.4f", train_accuracy)
    logger.info("test accuracy: %.4f", test_accuracy)
    logger.info("model saved to %s", output_path)

    return {
        "train_accuracy": train_accuracy,
        "test_accuracy": test_accuracy,
        "random_state": random_state,
        "hyperparameters": hyperparams,
        "n_train": len(x_train),
        "n_test": len(x_test),
    }


def main() -> None:
    parser = argparse.ArgumentParser(description="Train RandomForest on iris dataset")
    parser.add_argument(
        "--output-path",
        type=Path,
        default=Path("app/models/model.pkl"),
        help="Куда сохранить обученную модель",
    )
    parser.add_argument(
        "--random-state",
        type=int,
        default=42,
        help="Random seed для воспроизводимости",
    )
    parser.add_argument(
        "--metrics-path",
        type=Path,
        default=None,
        help="Опциональный JSON с метриками обучения",
    )
    args = parser.parse_args()

    metrics = train(args.output_path, args.random_state, DEFAULT_HYPERPARAMS)

    if args.metrics_path is not None:
        args.metrics_path.parent.mkdir(parents=True, exist_ok=True)
        args.metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()


Проверка работоспособности обучения

In [ ]:
!python ml_pipeline.py --output-path artifacts/model.pkl

Шаг `Make pipeline reproducible` сохраняет `pipeline_metadata.json`: pip freeze, git SHA, sha256 датасета и модели, random_state, гиперпараметры. Артефакт прикрепляется к run-у на 7 дней.

In [ ]:
%%writefile ci.yml
name: CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

concurrency:
  group: ci-${{ github.ref }}
  cancel-in-progress: true

permissions:
  contents: read
  packages: write

env:
  PYTHON_VERSION: "3.11"
  REGISTRY: ghcr.io
  IMAGE_NAME: ${{ github.repository }}

jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
      - uses: actions/cache@v4
        with:
          path: ~/.cache/pip
          key: pip-${{ runner.os }}-${{ hashFiles('requirements-dev.txt') }}
      - name: Install dev deps
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements-dev.txt
      - name: Ruff
        run: ruff check app/ tests/

  test:
    runs-on: ubuntu-latest
    needs: lint
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
      - uses: actions/cache@v4
        with:
          path: ~/.cache/pip
          key: pip-${{ runner.os }}-${{ hashFiles('requirements-dev.txt') }}
      - name: Install dev deps
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements-dev.txt
      - name: Pytest
        run: pytest tests/ -v --tb=short

  train:
    runs-on: ubuntu-latest
    needs: test
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
      - name: Install runtime deps
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt
      - name: Train and capture metrics
        run: |
          mkdir -p artifacts
          python -m app.ml_pipeline \
            --output-path artifacts/model.pkl \
            --random-state 42 \
            --metrics-path artifacts/metrics.json
      # Make pipeline reproducible.
      # На этом шаге фиксируется всё, что нужно для повторного запуска:
      #   - точные версии Python-пакетов (pip freeze)
      #   - SHA коммита (хеш кода)
      #   - sha256 датасета (хеш данных)
      #   - sha256 обученной модели (хеш артефакта)
      #   - random_state и гиперпараметры RandomForest
      - name: Make pipeline reproducible
        run: |
          pip freeze > artifacts/requirements_lock.txt
          python - <<'PY'
          import hashlib, json, os
          from sklearn.datasets import load_iris
          dataset_hash = hashlib.sha256(load_iris().data.tobytes()).hexdigest()
          with open("artifacts/model.pkl", "rb") as fh:
              model_hash = hashlib.sha256(fh.read()).hexdigest()
          with open("artifacts/metrics.json", "r", encoding="utf-8") as fh:
              metrics = json.load(fh)
          metadata = {
              "code_commit": os.environ.get("GITHUB_SHA", "local"),
              "run_id": os.environ.get("GITHUB_RUN_ID", "local"),
              "dataset_sha256": dataset_hash,
              "model_sha256": model_hash,
              "random_state": metrics["random_state"],
              "hyperparameters": metrics["hyperparameters"],
              "train_accuracy": metrics["train_accuracy"],
              "test_accuracy": metrics["test_accuracy"],
          }
          with open("artifacts/pipeline_metadata.json", "w", encoding="utf-8") as fh:
              json.dump(metadata, fh, indent=2, ensure_ascii=False)
          print(json.dumps(metadata, indent=2, ensure_ascii=False))
          PY
      - uses: actions/upload-artifact@v4
        with:
          name: pipeline-metadata
          path: artifacts/
          retention-days: 7

  build-image:
    runs-on: ubuntu-latest
    needs: test
    if: github.ref == 'refs/heads/main'
    steps:
      - uses: actions/checkout@v4
      - uses: docker/setup-buildx-action@v3
      - name: Login to GHCR
        uses: docker/login-action@v3
        with:
          registry: ${{ env.REGISTRY }}
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}
      - name: Build and push image
        uses: docker/build-push-action@v5
        with:
          context: .
          file: docker/Dockerfile
          push: true
          tags: |
            ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:${{ github.sha }}
            ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:latest
          build-args: |
            MODEL_VERSION=v1.0.0
          cache-from: type=gha
          cache-to: type=gha,mode=max
      - name: Logout from GHCR
        if: always()
        run: docker logout ${{ env.REGISTRY }}


In [ ]:
%%writefile deploy.yml
name: Deploy

on:
  push:
    branches: [main]
    tags:
      - "v*"
  workflow_dispatch:
    inputs:
      model_version:
        description: "MODEL_VERSION для билда"
        required: false
        default: "v1.0.0"

concurrency:
  group: deploy-${{ github.ref }}
  cancel-in-progress: false

permissions:
  contents: read
  packages: write

env:
  REGISTRY: ghcr.io
  IMAGE_NAME: ${{ github.repository }}
  MODEL_VERSION: ${{ github.event.inputs.model_version || vars.MODEL_VERSION || 'v1.0.0' }}

jobs:
  build-and-push:
    runs-on: ubuntu-latest
    outputs:
      image_tag: ${{ steps.meta.outputs.image_tag }}
    steps:
      - uses: actions/checkout@v4
      - uses: docker/setup-buildx-action@v3

      - name: Login to GHCR
        uses: docker/login-action@v3
        with:
          registry: ${{ env.REGISTRY }}
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}

      - id: meta
        name: Resolve tags
        run: |
          short_sha=${GITHUB_SHA::7}
          echo "image_tag=${short_sha}" >> "$GITHUB_OUTPUT"
          echo "Building image with MODEL_VERSION=${MODEL_VERSION}"

      - name: Build and push image
        uses: docker/build-push-action@v5
        with:
          context: .
          file: docker/Dockerfile
          push: true
          tags: |
            ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:${{ steps.meta.outputs.image_tag }}
            ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:${{ env.MODEL_VERSION }}
            ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:latest
          build-args: |
            MODEL_VERSION=${{ env.MODEL_VERSION }}
          cache-from: type=gha
          cache-to: type=gha,mode=max

      - name: Logout from GHCR
        if: always()
        run: docker logout ${{ env.REGISTRY }}

  deploy:
    runs-on: ubuntu-latest
    needs: build-and-push
    environment:
      name: production
    steps:
      - uses: actions/checkout@v4

      # Заглушка под реальный cloud API.
      # В прод-сценарии тут идёт вызов k8s rollout / Yandex Cloud / SberCloud API
      # с использованием secrets.CLOUD_TOKEN. Сейчас оставлен лог с параметрами
      # деплоя чтобы было видно полную форму запроса.
      - name: Deploy via cloud API
        env:
          CLOUD_TOKEN: ${{ secrets.CLOUD_TOKEN }}
        run: |
          echo "[deploy] image=${REGISTRY}/${IMAGE_NAME}:${{ needs.build-and-push.outputs.image_tag }}"
          echo "[deploy] model_version=${MODEL_VERSION}"
          if [[ -z "${CLOUD_TOKEN:-}" ]]; then
            echo "[deploy] CLOUD_TOKEN secret is not set, skipping real API call"
          else
            echo "[deploy] CLOUD_TOKEN present, would POST rollout request"
          fi

      - name: Health check with retries
        id: health
        continue-on-error: true
        env:
          TARGET_URL: ${{ vars.DEPLOY_HEALTH_URL || 'http://localhost:8080/health' }}
        run: |
          set +e
          for attempt in 1 2 3 4 5; do
            echo "[health] attempt ${attempt} -> ${TARGET_URL}"
            if curl -fsS --max-time 5 "${TARGET_URL}" >/dev/null; then
              echo "[health] ok"
              exit 0
            fi
            sleep 10
          done
          echo "[health] all attempts failed"
          exit 1

      - name: Auto rollback on health failure
        if: steps.health.outcome == 'failure'
        env:
          CLOUD_TOKEN: ${{ secrets.CLOUD_TOKEN }}
        run: |
          echo "[rollback] health failed, triggering rollback"
          if [[ -n "${CLOUD_TOKEN:-}" ]]; then
            echo "[rollback] would POST rollback=true to cloud API"
          fi

      - name: Fail job if rollback happened
        if: steps.health.outcome == 'failure'
        run: exit 1


Копируем workflow-файлы в `.github/workflows/`

In [ ]:
!mkdir -p .github/workflows
!mv ci.yml ./.github/workflows/ci.yml
!mv deploy.yml ./.github/workflows/deploy.yml

Отправляем изменения в репозиторий

In [ ]:
!git add .github/workflows/ ml_pipeline.py
!git commit -m "ci(github): add GitHub Actions ci and deploy workflows"
!git log --oneline

### Какие секреты нужно добавить

Settings -> Secrets and variables -> Actions:

- `CLOUD_TOKEN` (secret) - токен cloud-провайдера для прод-деплоя
- `MODEL_VERSION` (variable) - версия модели для build-arg и тегов образа

`GITHUB_TOKEN` доступен автоматически - используется для push в GHCR.

### Ссылки на успешные runs GitHub Actions

- CI (lint, test, train, build): https://github.com/SergeiVolkhin/ML-HW_7/actions/runs/25689123899
- Deploy (build, push в GHCR, health-check): https://github.com/SergeiVolkhin/ML-HW_7/actions/runs/25689123863


## 6. Итоговое оформление

**Что оказалось простым.** Структура docker-compose и FastAPI с `lifespan`-хендлером - стандартные паттерны, ушло на их сборку и тесты 9 unit-тестов меньше часа. ADR в формате Michael Nygard тоже простой - четыре раздела, никаких инструментов кроме редактора.

**Что вызвало трудности.** Параметризация весов в `nginx split_clients` через envsubst - конфликт между `$nginx-переменными` и `${envsubst-переменными}`, решилось через приём `DOLLAR='$$'`. Также nginx не принимает `0%` как значение веса, поэтому для rollback пришлось завести отдельный статический конфиг без `split_clients`. Третья трудность - синтаксис исходного `.gitlab-ci.yml` в шаблоне был в формате Gitea Actions (`jobs`, `runs-on`, `uses`), пришлось полностью переписать под GitLab CI (`stages`, `image`, `script`).

**Выводы по стратегии деплоя.** Canary - оптимальный выбор когда новая модель не была обкатана на проде и в коде сервиса отсутствует system-wide error handling: даже при необработанном исключении пострадает 10% запросов, не 100%. Цена - удвоение мониторинга (нужно разделять метрики по `X-Service-Version`) и stateless-требование к сервису (sticky-сессии конфликтуют с per-request хешем `$request_id` в ключе `split_clients`). Rollback за 439 мс на локальной машине - лучший аргумент в пользу выбора этой стратегии.